# STAN AI — Cloud GPU Training Workflow (Colab & Kaggle)

Train STAN AI's independent **51,658,240-parameter** decoder-only Transformer completely from random initialization:
1. Clone repository & install dependencies reproducibly
2. Verify CUDA GPU hardware
3. Prepare verified public-domain English corpus & train custom 8,192 SentencePiece BPE tokenizer
4. Stage 1: English Next-Token Pretraining (with automatic `--resume auto` checkpointing to Drive/Kaggle Output)
5. Stage 2: Supervised Instruction Tuning (SFT on Apache-2.0 `OpenAssistant/oasst1`)
6. Run 9-dimension evaluation & export completed checkpoint artifacts (`latest.pt`, `tokenizer.model`, `tokenizer_config.json`, `model_config.yaml`, `training_history.json`) to import back into STAN AI

In [ ]:
# 1. Clone STAN AI repository
!git clone https://github.com/ebronabiy12-ui/Stan-ai.git stan-ai-workspace || true
%cd stan-ai-workspace

In [ ]:
# 2. Storage directory persistence (Colab Drive or Kaggle Working directory)
import os
import sys
is_colab = os.path.exists('/var/colab/hostname') or 'google.colab' in sys.modules
if is_colab:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CKPT_DIR = '/content/drive/MyDrive/STAN_AI_Checkpoints'
    except Exception as e:
        print('Colab drive mount skipped:', e)
        CKPT_DIR = '/content/stan_ai_checkpoints'
else:
    # Kaggle persistent working directory
    CKPT_DIR = '/kaggle/working/stan_ai_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoint storage directory:', CKPT_DIR)

In [ ]:
# 3. Install dependencies reproducibly and verify CUDA GPU
!bash STAN-AI/scripts/colab_kaggle_setup.sh
!pip install -q datasets

In [ ]:
# 4. Stage 1 Corpus Preparation & 8,192 BPE Tokenizer (train split ONLY)
!python STAN-AI/scripts/prepare_corpus.py --sources-limit 12 --vocab-size 8192 --checkpoint-dir "$CKPT_DIR"

In [ ]:
# 5. Stage 1: English Pretraining from Random Initialization (auto-resumes if interrupted)
!python STAN-AI/scripts/kaggle_train.py --checkpoint-dir "$CKPT_DIR" --resume auto --max-steps 5000

In [ ]:
# 6. Stage 2: Prepare Apache-2.0 OpenAssistant OASST1 Instruction Corpus & Run SFT
!python STAN-AI/scripts/prepare_sft_corpus.py --output STAN-AI/data/processed/sft_instructions.jsonl --max-pairs 8000
!python STAN-AI/training/sft.py --pretrained-checkpoint "$CKPT_DIR/latest.pt" --tokenizer "$CKPT_DIR/tokenizer.model" --checkpoint-dir "$CKPT_DIR" --overwrite

In [ ]:
# 7. Run Real 9-Dimension Evaluation & Generate Export Bundle
!python STAN-AI/scripts/run_evaluation.py --checkpoint-dir "$CKPT_DIR"
!cp -r "$CKPT_DIR"/* /kaggle/working/ 2>/dev/null || true
!ls -lh "$CKPT_DIR"
# Download latest.pt + tokenizer.model from CKPT_DIR and import into STAN AI via Training Dashboard or:
# python STAN-AI/scripts/import_checkpoint.py --source-checkpoint /path/to/latest.pt --tokenizer /path/to/tokenizer.model --confirm-overwrite